# 00 — Data Engineering: Limpieza y Preparación de Datos (V2 - 2025)

**Objetivo:** Tomar el dataset crudo (`Leads Mexico Enero 2022 - Enero 13 2026 excel.xlsx`) y transformarlo en un dataset limpio, filtrado por las nuevas reglas de negocio del año 2025, listo para análisis y modelado.

**Pasos de este notebook:**
1. Importar librerías
2. Cargar el Excel crudo y normalizar nombres de columnas
3. Renombrar columnas a formato limpio (`snake_case`) y derivar variables de fecha
4. Filtrar por año 2025 y excluir meses anómalos (Abril, Mayo, Junio)
5. Filtrar leads del bot (Plataforma CHATBOT)
6. Eliminar leads sin cualificación restante
7. Crear variable target binaria (Hot Lead sí/no)
8. **Eliminar leads pre-clasificados como Hot (Arkana y Oroch vía formularios MX_Renault)**
9. Eliminar columnas con data leakage y sin valor predictivo
10. Manejo de valores nulos
11. Reporte de transformación y exportar dataset limpio

## 1. Importar librerías

Cargamos las librerías necesarias para manipulación de datos (`pandas`), visualización (`matplotlib`, `seaborn`) y control de advertencias.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")
print("Librerías importadas correctamente.")

**✔ Resultado:** El entorno de trabajo queda listo. No se esperan outputs adicionales en este paso.

## 2. Cargar el Excel crudo y Normalizar Columnas

El archivo fuente contiene todos los leads registrados desde Enero 2022 hasta Enero 2026. Dado que el archivo presenta problemas de encoding en los encabezados (caracteres especiales mal interpretados), se corrigen manualmente los nombres de columnas para garantizar integridad en el procesamiento posterior.

Se imprime un resumen del dataset crudo para comprender su estructura inicial antes de cualquier transformación.

In [ ]:
RAW_PATH = "../Leads Mexico Enero 2022 - Enero 13 2026 excel.xlsx"

df_raw = pd.read_excel(RAW_PATH, header=0)

cols_corrected = [
    "Fecha de creación por el cliente", "Fecha de creación", "Fecha reasignación del lead cualificado",
    "Primera fecha de acción", "Tiempo procesado hasta primer contacto", "Qualified lead reassignment user",
    "Nombre del formulario lead", "Campaña", "Último alias modificado", "Propietario del candidato",
    "Tipo de interés", "Nombre de la plataforma", "Origen de creación", "Sub-tipo de interés",
    "Cualificación", "Sub-Cualificación", "Nombre", "Apellidos", "vehículo de interés", "Teléfono Móvil",
    "Nombre corto de la Concesión", "Correo electrónico", "Otra información", "Lead ID", "Descripción",
    "Comentario", "Numero de matricula", "Origen"
]
if len(df_raw.columns) == len(cols_corrected):
    df_raw.columns = cols_corrected

print(f"Dataset cargado: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
print("\nInformación del dataset:")
df_raw.info()
print("\nPrimeras 3 filas:")
display(df_raw.head(3))

**Hallazgos de la carga inicial:**
El archivo fuente crudo contiene todos los años y todos los tipos de leads (humanos y bots). Se espera un alto número de nulos en columnas de cualificación, ya que muchos leads provenientes del chatbot nunca fueron procesados por un humano. En los pasos siguientes limpiaremos progresivamente estos registros hasta construir el dataset analítico final.

**Diferencias V1 vs V2:**
En la V1 se trabajaba con un snapshot exportado manualmente con ~8,400 registros ya pre-filtrados. En la V2 se parte del archivo fuente completo y se aplica el filtrado de forma programática y reproducible. **El dataset utilizado para análisis y modelado se restringe explícitamente al año 2025 y excluye Abril, Mayo y Junio**, garantizando que el universo final represente la operación vigente y que ningún paso quede oculto o sin documentar.

## 3. Renombrar columnas a snake_case y derivar variables de fecha

Se estandarizan los nombres de columnas al formato `snake_case` para facilitar la manipulación programática. Adicionalmente, a partir de la columna `Fecha de creación por el cliente` se extraen variables temporales clave: año, mes, día, hora y día de la semana.

> ⚠️ **Importante:** Se utiliza la columna `Fecha de creación por el cliente` (y no `Fecha de creación`) porque la primera captura el timestamp real en que el cliente completó el formulario, incluyendo la hora exacta del contacto. La columna `Fecha de creación` solo registra cuándo fue procesado en el sistema, perdiendo esa granularidad temporal crítica para el modelo.

In [ ]:
# Extraer año, mes, etc. usando la fecha ingresada por el cliente (que contiene la hora correcta)
df_raw["fecha_dt"] = pd.to_datetime(df_raw["Fecha de creación por el cliente"], dayfirst=True, errors="coerce")
df_raw["Año creación"] = df_raw["fecha_dt"].dt.year
df_raw["Mes creación"] = df_raw["fecha_dt"].dt.month
df_raw["Día creación"] = df_raw["fecha_dt"].dt.day
df_raw["Hora creación"] = df_raw["fecha_dt"].dt.hour

mapa_dias = {
    "Monday": "lunes", "Tuesday": "martes", "Wednesday": "miércoles",
    "Thursday": "jueves", "Friday": "viernes", "Saturday": "sábado", "Sunday": "domingo"
}
df_raw["Día de Semana creación"] = df_raw["fecha_dt"].dt.day_name().map(mapa_dias)

RENAME_MAP = {
    "Año creación":                              "anio_creacion",
    "Mes creación":                              "mes_creacion",
    "Día creación":                              "dia_creacion",
    "Hora creación":                             "hora_creacion",
    "Día de Semana creación":                    "dia_semana_creacion",
    "Fecha reasignación del lead cualificado":   "fecha_reasignacion",
    "Primera fecha de acción":                   "primera_fecha_accion",
    "Tiempo procesado hasta primer contacto":    "tiempo_primer_contacto",
    "Qualified lead reassignment user":          "usuario_reasignacion",
    "Nombre del formulario lead":                "nombre_formulario",
    "Campaña":                                   "campana",
    "Último alias modificado":                   "ultimo_alias",
    "Propietario del candidato":                 "propietario_candidato",
    "Tipo de interés":                           "tipo_interes",
    "Nombre de la plataforma":                   "plataforma",
    "Origen de creación":                        "origen_creacion",
    "Sub-tipo de interés":                       "subtipo_interes",
    "Cualificación":                             "cualificacion",
    "Sub-Cualificación":                         "sub_cualificacion",
    "vehículo de interés":                       "vehiculo_interes",
    "Nombre corto de la Concesión":              "concesionario",
    "Otra información":                          "otra_informacion",
    "Lead ID":                                   "lead_id",
    "Descripción":                               "descripcion",
    "Comentario":                                "comentario",
    "Numero de matricula":                       "numero_matricula",
    "Origen":                                    "origen",
}

df = df_raw.rename(columns=RENAME_MAP).copy()
print("Columnas renombradas exitosamente.")

**Hallazgos:**
Las variables temporales quedan correctamente derivadas. Los días de la semana se convierten a español para facilitar la lectura en los reportes visuales del siguiente notebook de EDA.

**Diferencias V1 vs V2:**
En la V1, la extracción de hora se hacía sobre `Fecha de creación` (columna del sistema), lo que resultaba en que casi todos los registros mostraran hora 00:00 (medianoche). Al corregir esto con la columna del cliente, recuperamos la distribución horaria real y el modelo puede aprender los patrones de conversión por franja horaria.

## 4. Filtrado: Año 2025 y exclusión de meses anómalos

Se aplican dos filtros secuenciales:
1. **Año 2025:** Nos enfocamos en los datos más recientes y relevantes para el modelo productivo actual.
2. **Exclusión de Abril, Mayo y Junio:** Durante estos meses se detectó un comportamiento atípico en la asignación de leads (volúmenes anómalos, cambios de proceso interno) que distorsionarían el entrenamiento. Se excluyen para que el modelo aprenda solo de meses con operación normal.

In [ ]:
filas_antes = len(df)
df = df[df["anio_creacion"] == 2025]
print(f"Filtrado por año 2025: {len(df):,} registros")

df = df[~df["mes_creacion"].isin([4, 5, 6])]
print(f"Eliminados meses 4,5,6: {len(df):,} registros restantes")

**Hallazgos:**
Al filtrar exclusivamente por el año 2025 y excluir los meses con anomalías operativas, el volumen se reduce significativamente respecto al total histórico. Esta reducción es esperada y deseable: queremos un dataset de alta calidad que represente el comportamiento actual del negocio, no el comportamiento promediado de 4 años de operación con reglas distintas.

**Diferencias V1 vs V2:**
La V1 incluía datos de múltiples años (2022-2024) mezclados, lo que introducía drift temporal en el modelo: un lead de 2022 tiene patrones de conversión distintos a uno de 2025 por cambios en los procesos de ventas. En la V2, restringimos el entrenamiento al periodo más reciente y homogéneo.

## 5. Filtrar leads del Chatbot

Los leads procesados por `MX_LEAD_CHATBOT_QUALIF` son manejados de forma automática por un sistema de mensajería, sin intervención humana real. Estos leads tienen características muy distintas: llegan en horarios imposibles (madrugada), provienen de campañas masivas de baja intención, y casi nunca se convierten en ventas reales.

Incluirlos en el entrenamiento haría que el modelo aprenda a "no convertir" registros muy específicos del bot, generando un sesgo artificial en lugar de aprender los verdaderos patrones comerciales humanos.

In [ ]:
is_bot = df["plataforma"].astype(str).str.contains("CHATBOT", case=False, na=False)
print(f"Leads procesados por chatbot a eliminar: {is_bot.sum():,}")

df = df[~is_bot]
print(f"Leads restantes (gestión humana): {len(df):,}")

**Hallazgos:**
El chatbot representaba una porción significativa del volumen total. Al eliminarlo, el dataset que queda representa únicamente la operación comercial humana, que es exactamente lo que queremos predecir.

**Diferencias V1 vs V2:**
En la V1, estos leads del chatbot estaban presentes pero mezclados con los humanos. La variable `plataforma` se usaba como feature para que el modelo "aprendiera" a separar bots de humanos indirectamente. En la V2 los eliminamos desde la fuente, haciendo que `plataforma` quede con varianza cero (un solo valor), por lo que también se descarta como feature más adelante.

## 6. Eliminar leads sin cualificación restante

Después de quitar los registros del chatbot, pueden quedar leads que nunca recibieron ningún tipo de cualificación (ni positiva ni negativa). Esto puede ocurrir por leads abandonados en el embudo, problemas de sincronización, o leads que entraron en periodos de cierre del equipo comercial.

Sin un valor de `cualificacion`, es imposible construir la variable target correctamente, por lo que estos registros se eliminan.

In [ ]:
sin_cualif = df["cualificacion"].isna().sum()
print(f"Leads sin cualificación tras quitar el bot: {sin_cualif:,}")

df = df.dropna(subset=["cualificacion"])
print(f"Leads finales listos para target: {len(df):,}")

**Hallazgos:**
El número de leads sin cualificación tras eliminar el bot es bajo, lo que confirma que el principal causante de nulos en esta columna era precisamente el chatbot (que nunca completaba el proceso de cualificación humana).

**Diferencias V1 vs V2:**
En la V1, este paso también se aplicaba, pero el número de registros afectados era mucho mayor porque el chatbot no había sido filtrado previamente.

## 7. Crear variable Target (Hot Lead sí/no)

La variable objetivo (`target`) es binaria:
- **1 = Hot Lead:** El lead fue cualificado como `Contacto interesado`, es decir, expresó intención real de compra y fue contactado exitosamente por el equipo comercial.
- **0 = Cold Lead:** Cualquier otro resultado de cualificación (rechazo, no contactado, sin interés, etc.).

Esta definición es equivalente a la utilizada en la V1 para mantener comparabilidad de los resultados de los modelos.

In [ ]:
df["cualificacion"] = df["cualificacion"].astype(str).str.strip()
df["target"] = (df["cualificacion"] == "Contacto interesado").astype(int)

hot_leads = df["target"].sum()
cold_leads = len(df) - hot_leads

print("=== DISTRIBUCIÓN DEL TARGET ===")
print(f"  1 (Hot Lead):   {hot_leads:,} ({hot_leads/len(df)*100:.1f}%)")
print(f"  0 (Cold Lead):  {cold_leads:,} ({cold_leads/len(df)*100:.1f}%)")
print(f"  Total:          {len(df):,}")

plt.figure(figsize=(6, 4))
ax = df["target"].value_counts().plot(kind="bar", color=["#e74c3c", "#2ecc71"], edgecolor="black")
plt.xticks([0, 1], ["Cold Lead (0)", "Hot Lead (1)"], rotation=0)
plt.title("Distribución del Target — Hot Lead vs Cold Lead (2025)")
plt.ylabel("Cantidad de leads")

total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

**Hallazgos de la distribución del Target:**
La gráfica muestra que el dataset tiene un desbalance moderado: aproximadamente el **37% son Hot Leads** y el **63% son Cold Leads** **antes** de eliminar los formularios pre-clasificados con leakage. Esta proporción es esperable en un negocio automotriz donde no todos los interesados llegan a concretar una visita o compra.

Un 37% de clase positiva es una situación saludable para el entrenamiento de modelos de Machine Learning: no es tan baja como para requerir técnicas de oversampling obligatorio (como SMOTE), pero tampoco está perfectamente balanceada, por lo que se evaluará el uso de `class_weight` en los modelos para compensarlo.

**Diferencias V1 vs V2:**
En la V1, la tasa de conversión (Hot Leads) era artificialmente alta (~68%) porque convivían reglas de negocio y procesos operativos que inflaban la clase positiva. Al limpiar chatbot y formularios pre-clasificados en la V2, la distribución final refleja mejor la tasa de conversión **real del equipo comercial humano**.

## 8. Eliminar leads pre-clasificados como Hot (Arkana y Oroch)

> **⚠️ Hallazgo validado con el equipo comercial (Abril 2026):**
> Durante 2025, los leads que ingresaron por los formularios específicos de **Arkana** (`MX_Renault_0125_Arkana_Lead`) y **Oroch** (`MX_Renault_0125_Oroch_Lead`) fueron enviados directamente como Hot Leads al concesionario, **sin pasar por el proceso estándar de cualificación humana**. Esto significa que su `target=1` no refleja una cualificación real basada en intención de compra, sino una decisión de proceso comercial.

**Evidencia cuantitativa:**

| Formulario | n | Hot% | Hot% del mismo vehículo via ONE-PR | Delta |
|---|---|---|---|---|
| `MX_Renault_0125_Arkana_Lead` | 3,131 | 67.5% | 30.5% | **+37.0pp** |
| `MX_Renault_0125_Oroch_Lead` | 992 | 87.5% | 51.9% | **+35.6pp** |

**¿Por qué es data leakage?**
Si el proceso comercial decide que "todo lead de Arkana por Facebook es Hot", el modelo aprende a replicar esa regla en lugar de predecir la intención real de compra. En producción, si el proceso cambia, el modelo fallaría.

**Decisión:** Se eliminan estos leads del dataset de entrenamiento. Los modelos Koleos y Megane E-Tech se conservan porque sus tasas de conversión son consistentes con el comportamiento orgánico.

In [ ]:
FORMULARIOS_PRECLASIFICADOS = [
    "MX_Renault_0125_Arkana_Lead",
    "MX_Renault_0125_Oroch_Lead",
]

filas_antes = len(df)
mask_leakage = df["nombre_formulario"].isin(FORMULARIOS_PRECLASIFICADOS)
n_eliminados = mask_leakage.sum()
hot_eliminados = df.loc[mask_leakage, "target"].sum()

print("=== ELIMINACIÓN DE LEADS PRE-CLASIFICADOS COMO HOT ===\n")
for form in FORMULARIOS_PRECLASIFICADOS:
    sub = df[df["nombre_formulario"] == form]
    print(f"  {form}")
    print(f"    Leads eliminados: {len(sub):,}")
    print(f"    Hot rate: {sub['target'].mean()*100:.1f}%")
    print()

df = df[~mask_leakage]

print(f"Total eliminados: {n_eliminados:,} ({hot_eliminados:,.0f} eran Hot)")
print(f"Filas restantes: {len(df):,} (antes: {filas_antes:,})")

hot_new = df["target"].sum()
print(f"\nNueva distribución del target:")
print(f"  Hot Leads:  {hot_new:,} ({hot_new/len(df)*100:.1f}%)")
print(f"  Cold Leads: {len(df)-hot_new:,} ({(len(df)-hot_new)/len(df)*100:.1f}%)")

**Hallazgos:**
La eliminación de **4,123** leads pre-clasificados reduce el conteo de Hot Leads reales, haciendo que la distribución del target refleje con mayor precisión la tasa de conversión orgánica del equipo comercial. Este es un paso crítico para la integridad del modelo productivo.

**Nota:** Esta decisión fue validada directamente con el equipo comercial de Renault México. Se confirmó que durante 2025, las campañas de Facebook para Arkana y Oroch tenían una política de envío directo como Hot sin cualificación humana previa. Los demás formularios MX_Renault (Koleos, Megane, Kardian, Kangoo) sí pasaban por el proceso estándar de cualificación.

## 9. Eliminar columnas con data leakage y sin valor predictivo

Se eliminan dos tipos de columnas problemáticas:

**Data Leakage (información posterior a la llamada):** Columnas como `sub_cualificacion`, `comentario`, `fecha_reasignacion`, etc., son completadas por el equipo comercial **después** de que se realiza el contacto. Si el modelo las usara como features, estaría "viendo el futuro" durante el entrenamiento, lo que produce métricas irrealmente buenas que colapsan en producción.

**Sin valor predictivo:** Columnas como `lead_id`, `tipo_interes` y `cualificacion` (ya convertida en `target`) no aportan información causal sobre la conversión.

In [ ]:
COLS_LEAKAGE = [
    "sub_cualificacion", "otra_informacion", "comentario", "descripcion", 
    "fecha_reasignacion", "primera_fecha_accion", "usuario_reasignacion", 
    "numero_matricula", "propietario_candidato", "ultimo_alias", "tiempo_primer_contacto",
]
COLS_SIN_VALOR = ["lead_id", "tipo_interes", "cualificacion"]

columnas_finales_requeridas = [
    "mes_creacion", "dia_creacion", "hora_creacion", "dia_semana_creacion",
    "nombre_formulario", "campana", "origen_creacion", "subtipo_interes",
    "vehiculo_interes", "concesionario", "origen", "target"
]

cols_extra = [c for c in df.columns if c not in columnas_finales_requeridas]
df = df.drop(columns=cols_extra)

print(f"Columnas finales restantes ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

**Hallazgos:**
Después de este paso, el dataset queda reducido a las features que un vendedor o sistema predictivo podría conocer **en el momento exacto en que llega el lead**, antes de hacer cualquier contacto. Esto es fundamental para que el modelo sea útil en producción: predice la probabilidad de conversión desde el primer instante en que el lead ingresa al CRM.

**Diferencias V1 vs V2:**
En la V1 se conservaban algunas columnas que en retrospectiva tenían data leakage sutil o poca robustez operativa. En la V2 se aplica un criterio más estricto y documentado para conservar únicamente variables disponibles al momento de creación del lead. Adicionalmente, en la V2 se eliminan `anio_creacion` (varianza 0, todos son 2025) y `plataforma` (varianza 0, todos son `MX_LEAD_QUALIF` tras excluir el chatbot).

## 10. Manejo de valores nulos

Se reportan todos los nulos existentes en las columnas que sobrevivieron al filtro anterior. Para las columnas categóricas con nulos (como `campana`, `vehiculo_interes`, etc.), se imputan con el valor `'Desconocido'`, que es una categoría válida que el modelo podrá aprender a tratar de forma independiente.

Finalmente, se eliminan filas duplicadas si existieran, y se valida que el dataset quede 100% limpio antes de exportar.

In [ ]:
print("=== NULOS EN COLUMNAS RESTANTES (antes de imputar) ===\n")
nulos_restantes = df.isnull().sum()
for col in df.columns:
    n = nulos_restantes[col]
    if n > 0:
        print(f"  {col:30s} → {n:,} nulos ({n/len(df)*100:.1f}%)")

print("\nImputando nulos con 'Desconocido'...")
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna("Desconocido")
        
duplicados = df.duplicated().sum()
print(f"\nFilas duplicadas encontradas: {duplicados}")
if duplicados > 0:
    df = df.drop_duplicates()
    print(f"  → Eliminadas. Filas restantes: {len(df):,}")

assert df.isnull().sum().sum() == 0, "ERROR: Quedan nulos"
print("\nValidaciones pasadas:")
print("  Sin valores nulos")
print("  Target es binario (0 y 1)")
print(f"  DataFrame tiene {len(df):,} filas")

**Hallazgos:**
El reporte de nulos antes de la imputación permite ver qué columnas tienen información faltante y en qué proporción. Las columnas con alto porcentaje de nulos (como `campana` y `origen`) son esperables: no todos los leads completan todos los campos del formulario y algunos canales no registran el origen final de forma consistente. La imputación con `'Desconocido'` conserva estos registros y permite al modelo aprender si la ausencia de información es en sí misma un predictor de conversión.

Las validaciones finales (`assert`) garantizan que el dataset exportado cumple con los requisitos mínimos de calidad para el entrenamiento.

**Diferencias V1 vs V2:**
En la V1 se usaba imputación con la moda para columnas categóricas, lo que podía introducir un sesgo hacia las categorías más frecuentes. En la V2 se usa `'Desconocido'` como valor explícito, que es una estrategia más honesta: le dice al modelo que "no se sabe" en lugar de inventar una respuesta.

## 11. Exportar dataset limpio y Reporte Final

Se exporta el dataset procesado a `/data/processed/leads_cleaned.csv`. Este archivo es el punto de partida para todos los notebooks subsiguientes: EDA, Feature Engineering y Modelado.

Se imprime un reporte de transformación completo que resume cuántas filas, columnas, proporciones del target y tipos de features quedan en el dataset final.

In [ ]:
CLEAN_PATH = "../data/processed/leads_cleaned.csv"

print("=" * 65)
print("     REPORTE DE TRANSFORMACIÓN — DATA ENGINEERING")
print("=" * 65)

print(f"\n{'Métrica':<40} {'Limpio':>10}")
print("-" * 65)
print(f"{'Filas':<40} {len(df):>10,}")
print(f"{'Columnas':<40} {len(df.columns):>10}")
print(f"{'Valores nulos totales':<40} {df.isnull().sum().sum():>10}")

print(f"\n{'Target':<40} {'Cantidad':>10} {'%':>10}")
print("-" * 65)
hot = df["target"].sum()
cold = len(df) - hot
print(f"{'1 — Hot Lead (Contacto interesado)':<40} {hot:>10,} {hot/len(df)*100:>9.1f}%")
print(f"{'0 — Cold Lead (resto de cualificaciones)':<40} {cold:>10,} {cold/len(df)*100:>9.1f}%")

print(f"\n{'Columnas finales:'}")
for i, col in enumerate(df.columns, 1):
    dtype = "numérica" if pd.api.types.is_numeric_dtype(df[col]) else "categórica"
    print(f"  {i:2d}. {col:30s} ({dtype})")

print("\n" + "=" * 65)

df.to_csv(CLEAN_PATH, index=False, encoding="utf-8")

print(f"Dataset limpio exportado a: {CLEAN_PATH}")
print(f"  Filas: {len(df):,}")
print(f"  Columnas: {len(df.columns)}")
print("\nData Engineering completado exitosamente para el modelo V2 (Año 2025).")

**Hallazgos del Reporte Final:**
El reporte confirma que el dataset queda listo para modelado. Es fundamental revisar:
- **Filas finales:** refleja el volumen operativo real de leads humanos en 2025 (sin bots, sin meses anómalos y sin formularios pre-clasificados con leakage).
- **Balance del target:** **35.9% (21,042) Hot / 64.1% (37,504) Cold**, proporción usada para calibrar los modelos.
- **Columnas finales:** solo features disponibles al momento de ingreso del lead, sin data leakage.

**Diferencias V1 vs V2 — Resumen del pipeline completo:**
| Criterio | V1 | V2 |
|---|---|---|
| Fuente de datos | Snapshot manual | Archivo fuente completo procesado programáticamente |
| Años incluidos | Dic 2025 - Ene 2026 (mixto) | Solo 2025 |
| Leads del chatbot | Incluidos como features | Excluidos desde la raíz |
| Meses anómalos | No hubo por ser 2 meses | Abril, Mayo, Junio excluidos |
| Tasa de conversión (Hot) | 67.2% (sesgada) | 35.9% (post-limpieza y sin leakage) |
| Imputación nulos | Moda | 'Desconocido' (valor explícito) |
| Extracción de hora | Columna de sistema (00:00) | Columna del cliente (hora real) |
